In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica pelo método de Park (1999)
+ Aplicação global em toda a base (sem e com falha)
+ Classificação de falhas (RandomForestClassifier)
+ Split por temperatura (sem overlap)
+ RF fortemente regularizado para evitar overfitting
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"
REF_TEMP = 20
FREQ_MIN_KHZ = 35
FREQ_MAX_KHZ = 45
SMOOTH_WIN = 5

# Park parameters
PARK_MAX_SHIFT_FRAC = 0.25
PARK_OVERLAP_MIN = 0.60
PARK_SMOOTH_WIN = 5
PARK_NSTEPS = 201

# Random Forest (fortemente limitado)
RF_CLASSIF_PARAMS = dict(
    n_estimators=150,             
    max_depth=4,                 # árvore rasa, mas ainda capaz
    min_samples_split=20,        # evita dividir demais
    min_samples_leaf=8,          # folhas moderadas
    max_features=0.25,           # 25% das features → bom equilíbrio
    bootstrap=True,
    max_samples=0.70,            # 70% das curvas por árvore → anti-overfitting
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# ========= FUNÇÕES AUXILIARES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def moving_average(arr, win):
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode='edge')
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode='valid')
    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr)-len(smooth)), mode='edge')
    return smooth

# ========= MÉTODO DE PARK (1999) =========
def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=PARK_MAX_SHIFT_FRAC,
                           overlap_min_frac=PARK_OVERLAP_MIN,
                           smooth_win=PARK_SMOOTH_WIN,
                           nsteps=PARK_NSTEPS):
    """Compensação térmica de Park: busca τ e ΔS que minimizam o erro quadrático."""
    n = len(x)
    fmin, fmax = fhz[0], fhz[-1]
    df_band = fmax - fmin
    tau_max = max_shift_frac * df_band

    tau_vals = np.linspace(-tau_max, tau_max, nsteps)
    best = (np.inf, 0.0, 0.0)  # (erro mínimo, tau, deltaS)

    for tau in tau_vals:
        x_shift = shift_interp(x, fhz, tau)
        xs = x_shift; yr = y_ref
        if len(xs) < int(overlap_min_frac * n):
            continue
        deltaS = float(np.mean(yr - xs))
        resid = yr - (xs + deltaS)
        Va = float(np.sum(resid * resid))
        if Va < best[0]:
            best = (Va, tau, deltaS)

    _, tau_best, dS_best = best
    yout = shift_interp(x, fhz, tau_best) + dS_best
    if smooth_win > 1 and smooth_win % 2 == 1:
        yout = moving_average(yout, smooth_win)
    return yout, tau_best, dS_best

def park_batch(X, y_ref, fhz):
    n, m = X.shape
    Y = np.zeros_like(X)
    taus, deltas = [], []
    for i in range(n):
        yi, tau, dS = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = yi
        taus.append(tau)
        deltas.append(dS)
    return Y, np.array(taus), np.array(deltas)

# ========= ETAPA 1 – CARREGAMENTO =========
print("🔹 Carregando base completa...")
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz / 1e3
print(f"Nº amostras: {len(df)} | Nº features: {len(fcols)}")

# ========= ETAPA 2 – REFERÊNCIA SEM FALHA @20°C =========
df_sem = df[df["falha"] == 0].copy()
pool_ref = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
if len(pool_ref) == 0:
    print("⚠️ Nenhuma curva sem falha exata a 20°C — usando mediana global sem falha.")
    y_ref = np.median(df_sem[fcols].to_numpy(float), axis=0)
else:
    y_ref = np.median(pool_ref, axis=0)
print(f"Referência calculada com {len(pool_ref)} curvas @ {REF_TEMP}°C.")

# ========= ETAPA 3 – COMPENSAÇÃO DE PARK =========
print("\n🔹 Aplicando compensação térmica (Park, 1999)...")
X_all = df[fcols].to_numpy(float)
t0 = time.time()
Y_comp, taus, deltas = park_batch(X_all, y_ref, fhz)
print(f"✅ Compensação concluída em {time.time()-t0:.1f}s")

df_comp = df.copy()
df_comp[fcols] = Y_comp

# ========= ETAPA 4 – SPLIT POR TEMPERATURA =========
temps_all = sorted(df_comp["temperatura_c"].unique())
temps_train = temps_all[::2]
temps_test  = temps_all[1::2]

print(f"\nTemperaturas treino: {temps_train}")
print(f"Temperaturas teste:  {temps_test}")

df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)].copy()
df_test  = df_comp[df_comp["temperatura_c"].isin(temps_test)].copy()

X_train = df_train[fcols].to_numpy(float)
y_train = df_train["falha"].to_numpy(int)
X_test  = df_test[fcols].to_numpy(float)
y_test  = df_test["falha"].to_numpy(int)

print(f"Amostras treino: {len(X_train)} | teste: {len(X_test)}")

# ========= ETAPA 5 – CLASSIFICAÇÃO =========
print("\n🔹 Treinando RandomForestClassifier (RF regularizado)...")
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n== RESULTADOS RANDOM FOREST (Park, RF regularizado, split térmico) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))


print("\n✅ Execução concluída — RF limitado para evitar overfitting.")


In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.spatial.distance import cosine
from math import acos, degrees

def calc_metrics(y_true, y_pred):
    """Calcula todas as métricas entre curvas"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0,1]
    # SAM (Spectral Angle Mapper)
    sam_rad = acos(np.clip(np.dot(y_true, y_pred) /
                           (np.linalg.norm(y_true) * np.linalg.norm(y_pred) + 1e-12), -1, 1))
    sam_deg = degrees(sam_rad)
    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)
    rmsd = np.sqrt(np.mean((y_true - y_pred - np.mean(y_true - y_pred))**2))
    ccdm = 1 - corr
    return dict(R2=r2, RMSE=rmse, MAE=mae, Corr=corr,
                SAM_deg=sam_deg, NRMSE=nrmse, RMSD=rmsd, CCDM=ccdm)

# ===== Exemplo de uso =====
# y_ref: referência 20°C (ex: y_ref do código principal)
# X_orig: curva original (ex: df.loc[idx_show, fcols])
# X_comp: curva compensada (ex: df_comp.loc[idx_show, fcols])

y_ref_vec = y_ref
X_orig = df.loc[idx_show, fcols].to_numpy(float)
X_comp = df_comp.loc[idx_show, fcols].to_numpy(float)

print("\n== MÉTRICAS ORIGINAL vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_orig))

print("\n== MÉTRICAS COMPENSADO vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_comp))


In [ ]:
plt.rcParams.update({
    'font.size': 20,
    'text.usetex': True,
    'font.family': "Times New Roman"
})

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,5))

# Curva principal
ax.plot(y*1e3, F, '-', lw=2, color=colors[-1], label=r'$F$')

# ===== NOVOS MARCADORES (substituindo círculos e quadrados) =====
# Triângulos vazados apontando para cima
value = 5.08  # antes 9.805 se quiser voltar
ax.plot([-value], [0], marker='^', ms=14, mew=2.5, fillstyle='none', color='c')
ax.plot([ value], [0], marker='^', ms=14, mew=2.5, fillstyle='none', color='c')

# Triângulo central apontando para baixo
ax.plot([0], [0], marker='v', ms=14, mew=2.5, fillstyle='none', color='g')

# EIXOS
ax.set_ylabel(r'Force [N]', fontsize=20, rotation=90, labelpad=20)
ax.set_xlabel(r'$y_c$ [mm]', fontsize=20, rotation=0, labelpad=20)

ax.set_xlim([-7, 7])
ax.set_ylim([-5, 5])

ax.grid(True)
plt.show()

fig.tight_layout()